# Hackathon Data Access

This notebook gets you set up with the hackathon datasets. It handles:
- Installing dependencies
- Configuring credentials
- Browsing available datasets
- Downloading data with resume support and checksum verification

Works on: **Colab**, **Kaggle**, **RunPod**, and **local machines**.

## 1. Install dependencies

In [ ]:
!pip install -q boto3 tqdm

## 2. Get the helper module

If you're on Colab/Kaggle and don't have `r2_download.py` locally, this cell downloads it.

In [ ]:
import os
from pathlib import Path

HELPER_URL = "REPLACE_WITH_RAW_URL"  # organizer: set this to the raw URL of r2_download.py

if not Path("r2_download.py").exists():
    print("Downloading r2_download.py...")
    import urllib.request
    urllib.request.urlretrieve(HELPER_URL, "r2_download.py")
    print("Done.")
else:
    print("r2_download.py already present.")

## 3. Configure credentials

Set your R2 credentials. The organizers will provide these values.

In [ ]:
# Option A: Set directly (for quick testing)
os.environ["R2_ENDPOINT"] = "https://YOUR_ACCOUNT_ID.r2.cloudflarestorage.com"
os.environ["R2_ACCESS_KEY_ID"] = "YOUR_ACCESS_KEY"
os.environ["R2_SECRET_ACCESS_KEY"] = "YOUR_SECRET_KEY"
os.environ["R2_BUCKET"] = "hackathon-data"

# Option B: Load from .env file (if running locally)
# from dotenv import load_dotenv
# load_dotenv(".env")

## 4. Import helper and create client

In [ ]:
import r2_download as hd

print(f"Detected environment: {hd._detect_environment()}")
print(f"Default data directory: {hd._default_data_dir()}")

client = hd.get_s3_client()

## 5. Load manifest and browse datasets

In [ ]:
manifest = hd.load_manifest(
    bucket=os.environ["R2_BUCKET"],
    s3_client=client,
    cache_path="manifest.json",  # caches locally to avoid re-downloading
)

hd.summarize_manifest(manifest)

## 6. List shards for a specific dataset

In [ ]:
# Pick a dataset — replace with the one you want
DATASET = list(manifest["datasets"].keys())[0]

shards = hd.list_shards(manifest, dataset=DATASET)
print(f"Dataset: {DATASET}")
print(f"Shards: {len(shards)}")
print(f"Total size: {sum(s['size_bytes'] for s in shards) / 1e9:.2f} GB")
print()

# Show first few shards
for s in shards[:5]:
    print(f"  {s['key']}  ({s['size_bytes'] / 1e6:.1f} MB)  tags={s.get('tags', [])}")
if len(shards) > 5:
    print(f"  ... and {len(shards) - 5} more")

## 7. Download a dataset

Downloads with progress bars, resume support, and SHA-256 verification.
If interrupted, re-run the cell — it picks up where it left off.

In [ ]:
stats = hd.download_dataset(
    manifest,
    dataset_name=DATASET,
    # dest_dir="./my_data",  # uncomment to override default
    # tags=["train"],         # uncomment to download only specific tags
)

print(f"\nDownloaded: {stats['downloaded']}, Skipped: {stats['skipped']}, Failed: {stats['failed']}")
if stats["errors"]:
    print("Errors:")
    for e in stats["errors"]:
        print(f"  {e['key']}: {e['error']}")

## 8. Verify downloads

In [ ]:
# Re-run download with verify=True to check all existing files
# (skips downloading, just verifies checksums)
verify_stats = hd.download_dataset(
    manifest,
    dataset_name=DATASET,
    resume=True,
    verify=True,
)
print(f"All files verified: {verify_stats['failed'] == 0}")

## 9. Inspect the data

Load one shard and peek at the structure. Adjust based on the data format.

In [ ]:
from pathlib import Path

data_dir = Path(hd._default_data_dir())
first_shard = shards[0]["key"]
shard_path = data_dir / first_shard

print(f"Shard: {shard_path}")
print(f"Size: {shard_path.stat().st_size / 1e6:.1f} MB")

# === Uncomment the loader that matches your data format ===

# Parquet:
# import pandas as pd
# df = pd.read_parquet(shard_path)
# print(f"Shape: {df.shape}")
# df.head()

# CSV:
# import pandas as pd
# df = pd.read_csv(shard_path)
# print(f"Shape: {df.shape}")
# df.head()

# tar.gz (list contents):
# import tarfile
# with tarfile.open(shard_path, 'r:gz') as tar:
#     for member in tar.getmembers()[:10]:
#         print(f"  {member.name}  ({member.size} bytes)")

## 10. Next steps

You now have the data downloaded and verified. From here:

1. **Explore** the data — look at distributions, missing values, samples
2. **Build your pipeline** — data loading, preprocessing, feature engineering
3. **Train models** — start simple, iterate

Tips:
- If your download was interrupted, just re-run cell 7 — it resumes automatically
- Use `tags` parameter to download only the subset you need
- On Colab, data persists for the session. Re-download if you reconnect.
- On RunPod, data in `/workspace` persists across pod restarts.

---

# Precipitation Nowcasting — Data Exploration

The sections below walk you through the weather station and LDAS datasets. By the end you'll have a clear picture of what variables are available, where the data gaps are, and what precipitation patterns look like across the island.